# PyTorch IoU 与按类 NMS

输入框为连续坐标 `[x1, y1, x2, y2]`。IoU 是交集面积除以并集面积。广播可一次计算两组框的所有两两 IoU。

NMS 每次取最高分框，再筛掉 IoU 超过阈值的候选。按类 NMS 只抑制同一类别中的候选，最后按分数合并保留结果。

In [ ]:
import torch


def box_area(boxes):
    return (boxes[:, 2:] - boxes[:, :2]).clamp_min(0).prod(dim=-1)


def box_iou(boxes1, boxes2):
    left = torch.maximum(boxes1[:, None, :2], boxes2[:, :2])
    right = torch.minimum(boxes1[:, None, 2:], boxes2[:, 2:])
    intersection = (right - left).clamp_min(0).prod(dim=-1)
    union = box_area(boxes1)[:, None] + box_area(boxes2) - intersection
    return torch.where(union > 0, intersection / union, 0.0)


def nms_indices(boxes, scores, threshold):
    order = scores.argsort(descending=True, stable=True)
    keep = []
    while order.numel():
        selected, rest = order[:1], order[1:]
        keep.append(selected)
        order = rest[box_iou(boxes[selected], boxes[rest])[0] <= threshold]
    return torch.cat(keep) if keep else order


def batched_nms(boxes, scores, labels, threshold):
    keep = []
    for label in labels.unique():
        indices = torch.where(labels == label)[0]
        keep.append(indices[nms_indices(boxes[indices], scores[indices], threshold)])
    if not keep:
        return labels.new_empty((0,), dtype=torch.long)
    selected = torch.cat(keep).sort().values
    return selected[scores[selected].argsort(descending=True, stable=True)]


boxes = torch.tensor([[0, 0, 3, 3], [0, 0, 3, 3], [8, 8, 9, 9]], dtype=torch.float32)
scores = torch.tensor([0.9, 0.8, 0.7])
assert nms_indices(boxes, scores, 0.5).tolist() == [0, 2]
assert batched_nms(boxes, scores, torch.tensor([0, 1, 0]), 0.5).tolist() == [0, 1, 2]
assert batched_nms(boxes[:0], scores[:0], torch.empty(0, dtype=torch.long), 0.5).numel() == 0